# Module 12 — Positional Encoding

Here's a fact about attention (Modules 10-11) that's easy to miss:
**it has no idea what order the tokens came in.** Attention scores come
from `Q @ K.T` — dot products between content vectors — and the output is
a weighted sum of Value vectors. Nothing in that computation refers to
*position*. If you shuffled the input tokens around, attention would
produce the exact same set of outputs, just permuted along with the input.
That's fine for a "bag of tokens," but useless for language, where "dog
bites man" and "man bites dog" need to mean different things.

**Positional encoding** fixes this by injecting information about each
token's position directly into its embedding, before attention ever runs.

## 1. Proving attention is permutation-equivariant (has no sense of order)

We'll use non-causal attention here specifically to isolate this property —
causal masking depends on relative position too, which would muddy the
demonstration.

In [ ]:
import math

import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, causal=False):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


torch.manual_seed(42)
seq_len, d_model = 5, 8
x = torch.randn(seq_len, d_model)

Wq = torch.randn(d_model, d_model)
Wk = torch.randn(d_model, d_model)
Wv = torch.randn(d_model, d_model)

def attend(tokens):
    Q, K, V = tokens @ Wq, tokens @ Wk, tokens @ Wv
    out, _ = scaled_dot_product_attention(Q, K, V, causal=False)
    return out

out_original = attend(x)

perm = torch.randperm(seq_len)
x_permuted = x[perm]           # shuffle which token sits in which slot
out_permuted = attend(x_permuted)

# Claim: attending on the shuffled input, then un-shuffling the output,
# gives back the ORIGINAL output exactly - i.e. attention only cares about
# WHICH tokens are present, not what order they're in.
unshuffled = torch.zeros_like(out_permuted)
unshuffled[perm] = out_permuted

assert torch.allclose(unshuffled, out_original, atol=1e-5)
print("Confirmed: attend(shuffle(x)), un-shuffled, == attend(x) exactly.")
print("Attention alone cannot tell these two token orders apart.")

## 2. Sinusoidal positional encoding

The original Transformer paper's fix: build a fixed `(max_len, d_model)`
table where row `pos` is a vector of sines and cosines at different
frequencies. Even-indexed dimensions get `sin`, odd-indexed get `cos`, with
wavelengths increasing across the dimension — so nearby positions get
similar-but-distinguishable vectors, and far-apart positions look very
different.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    position = torch.arange(max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


pe = sinusoidal_positional_encoding(max_len=50, d_model=32)
print("positional encoding table shape:", pe.shape)

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.imshow(pe.T, cmap="RdBu", aspect="auto")
plt.xlabel("position")
plt.ylabel("embedding dimension")
plt.title("Sinusoidal positional encoding")
plt.colorbar()
plt.show()

## 3. Adding it in breaks the permutation-equivariance — on purpose

Now each token's embedding gets its slot's positional code added on top.
Repeat the exact same shuffle test as step 1: this time, un-shuffling the
output should **not** recover the original output, because the content
that ends up in a given slot now carries that slot's specific positional
signature — where it sits genuinely changes the result.

In [ ]:
pe_for_seq = sinusoidal_positional_encoding(max_len=seq_len, d_model=d_model)

x_with_pos = x + pe_for_seq
out_original_pos = attend(x_with_pos)

x_permuted_with_pos = x[perm] + pe_for_seq  # each slot still gets ITS OWN positional code
out_permuted_pos = attend(x_permuted_with_pos)

unshuffled_pos = torch.zeros_like(out_permuted_pos)
unshuffled_pos[perm] = out_permuted_pos

difference = (unshuffled_pos - out_original_pos).abs().max().item()
assert difference > 1e-3, "expected positional encoding to break permutation-equivariance"
print(f"Max difference after adding positional encoding: {difference:.4f} (nonzero, as expected)")
print("With positional information attached, moving a token to a different slot now genuinely changes the outcome.")

## 4. The alternative: learned positional embeddings

GPT-2 (and this project's nanoGPT in Module 17) use a simpler alternative:
just a plain `nn.Embedding(max_len, d_model)` indexed by position,
trained like any other embedding (Module 08) rather than fixed by a
sin/cos formula. It's less mathematically elegant but one line to
implement and lets the model learn whatever positional structure the data
actually needs.

In [ ]:
import torch.nn as nn

learned_pos_embedding = nn.Embedding(seq_len, d_model)
positions = torch.arange(seq_len)
learned_pe = learned_pos_embedding(positions)
print("learned positional embedding shape:", learned_pe.shape, "(same shape as the sinusoidal table, just trainable)")

## Recap

- Attention alone is permutation-equivariant — it has no concept of token
  order, only content.
- Positional encoding fixes this by adding a position-specific signal to
  each token's embedding before attention runs. We confirmed this
  concretely: without it, shuffling-then-attending-then-unshuffling exactly
  reproduces the original output; with it, that property breaks, because
  position now genuinely matters.
- Two common approaches: fixed sinusoidal encoding (original Transformer
  paper) or a learned embedding table (GPT-2 style, what Module 17's
  nanoGPT will use).

With attention (10-11) and positional encoding (12) in hand, Modules 13-15
build the remaining pieces — layer norm, residual connections, and the
feed-forward block — before Module 16 assembles a full transformer block.